In [59]:
import numpy as np 
import pandas as pd 
from sklearn.utils import resample 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('resume_data.csv')
df.shape

(9544, 35)

In [3]:
df.head()

,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,...,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],...,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],...,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],...,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667
3,NaN,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...","['Martinez Adult Education, Business Training ...",['Computer Applications Specialist Certificate...,['2008'],[None],[None],['Computer Applications'],"['Company Name ï¼ City , State', 'Company Name...",...,NaN,NaN,NaN,Business Development Executive,Bachelor/Honors,1 to 3 years,Age 22 to 30 years,Apparel Sourcing\nQuality Garment Sourcing\nRe...,Fast typing skill\nIELTSInternet browsing & on...,0.760000
4,NaN,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Kent State University'],['Bachelor of Business Administration'],[None],['3.84'],[None],['Accounting'],"['Company Name', 'Company Name', 'Company Name...",...,[None],[None],"['February 15, 2021']",Senior iOS Engineer,Bachelor of Science (BSc) in Computer Science,At least 4 years,NaN,iOS Lifecycle\nRequirement Analysis\nNative Fr...,iOS\niOS App Developer\niOS Application Develo...,0.650000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 35 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   address                              784 non-null    object 
 1   career_objective                     4740 non-null   object 
 2   skills                               9488 non-null   object 
 3   educational_institution_name         9460 non-null   object 
 4   degree_names                         9460 non-null   object 
 5   passing_years                        9460 non-null   object 
 6   educational_results                  9460 non-null   object 
 7   result_types                         9460 non-null   object 
 8   major_field_of_studies               9460 non-null   object 
 9   professional_company_names           9460 non-null   object 
 10  company_urls                         9460 non-null   object 
 11  start_dates                   

In [5]:
df.columns

Index(['address', 'career_objective', 'skills', 'educational_institution_name',
       'degree_names', 'passing_years', 'educational_results', 'result_types',
       'major_field_of_studies', 'professional_company_names', 'company_urls',
       'start_dates', 'end_dates', 'related_skils_in_job', 'positions',
       'locations', 'responsibilities', 'extra_curricular_activity_types',
       'extra_curricular_organization_names',
       'extra_curricular_organization_links', 'role_positions', 'languages',
       'proficiency_levels', 'certification_providers', 'certification_skills',
       'online_links', 'issue_dates', 'expiry_dates', '﻿job_position_name',
       'educationaL_requirements', 'experiencere_requirement',
       'age_requirement', 'responsibilities.1', 'skills_required',
       'matched_score'],
      dtype='object')

In [44]:
features = ["career_objective", "skills", "degree_names","responsibilities", "certification_skills", "role_positions"]
feature_df = df[features]
feature_df.head()

,career_objective,skills,degree_names,responsibilities,certification_skills,role_positions
0,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['B.Tech'],Technical Support\nTroubleshooting\nCollaborat...,NaN,NaN
1,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']",Machine Learning Leadership\nCross-Functional ...,NaN,NaN
2,NaN,"['Software Development', 'Machine Learning', '...",['B.Tech'],"Trade Marketing Executive\nBrand Visibility, S...",NaN,NaN
3,To obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...",['Computer Applications Specialist Certificate...,Apparel Sourcing\nQuality Garment Sourcing\nRe...,NaN,NaN
4,Professional accountant with an outstanding wo...,"['Analytical reasoning', 'Compliance testing k...",['Bachelor of Business Administration'],iOS Lifecycle\nRequirement Analysis\nNative Fr...,[None],"[None, None, None, None]"


In [7]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9544 entries, 0 to 9543
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   career_objective      4740 non-null   object
 1   skills                9488 non-null   object
 2   degree_names          9460 non-null   object
 3   responsibilities      9544 non-null   object
 4   certification_skills  2008 non-null   object
 5   role_positions        3426 non-null   object
dtypes: object(6)
memory usage: 447.5+ KB


In [24]:
feature_df.shape

(9544, 6)

In [34]:
null_values = ["None", "none", "[None]", "['']", "[]", "N/A", "nan", "NaN"]

In [21]:
feature_df['skills'].isna().sum()

np.int64(56)

In [23]:
feature_df['degree_names'].isna().sum()

np.int64(84)

In [42]:
(feature_df['degree_names'].isin(null_values)).sum()

np.int64(0)

In [40]:
(feature_df['responsibilities'].isin(null_values)).sum()

np.int64(0)

In [30]:
feature_df['certification_skills'].isna().sum()

np.int64(7536)

In [38]:
(feature_df['certification_skills'].fillna('').isin(null_values)).sum()

np.int64(832)

In [32]:
feature_df['role_positions'].isna().sum()

np.int64(6118)

In [39]:
(feature_df['role_positions'].fillna('').isin(null_values)).sum()

np.int64(168)

In [45]:
section_map = {
    "career_objective": "summary", 
    "skills": "skills",
    "degree_names": "degree",
    "responsibilities": "experience",
    "certification_skills": "certifications",
    "role_positions":"extracurricular"
}

In [46]:
def is_valid(text):
    null_values = ["None", "none", "[None]", "['']", "[]", "N/A", "nan", "NaN"]
    valid = True
    if text in null_values:
        valid = False 
    return valid 

In [47]:
pairs = []
for col, label in section_map.items():
    for val in df[col].dropna():
        if is_valid(val):
            pairs.append({'text': str(val).strip(), 'label': label})

In [48]:
len(pairs)

37610

In [49]:
training_df = pd.DataFrame(pairs)
training_df.shape

(37610, 2)

In [50]:
training_df.head()

,text,label
0,Big data analytics working and database wareho...,summary
1,Fresher looking to join as a data analyst and ...,summary
2,To obtain a position in a fast-paced business ...,summary
3,Professional accountant with an outstanding wo...,summary
4,"To secure an IT specialist, desktop support, n...",summary


In [51]:
training_df['label'].value_counts()

label
experience         9544
degree             9460
skills             9432
summary            4740
extracurricular    3258
certifications     1176
Name: count, dtype: int64

In [53]:
set(training_df['label'])

{'certifications',
 'degree',
 'experience',
 'extracurricular',
 'skills',
 'summary'}

In [55]:
max_samples = 2000 
min_samples = 500 
labels = set(training_df['label'])
balanced = []
for label in labels:
    subset = training_df[training_df['label'] == label]
    if len(subset) > max_samples:
        subset = resample(subset, n_samples=max_samples, random_state=42)
    elif len(subset) < min_samples:
        subset = resample(subset, n_samples=min_samples, random_state=42, replace=True)
    balanced.append(subset)

In [56]:
balanced_df = pd.concat(balanced).reset_index(drop=True)
balanced_df.shape

(11176, 2)

In [58]:
balanced_df['label'].value_counts()

label
skills             2000
experience         2000
degree             2000
summary            2000
extracurricular    2000
certifications     1176
Name: count, dtype: int64

In [61]:
le = LabelEncoder()
balanced_df['label_enc'] = le.fit_transform(balanced_df['label'])
balanced_df.head()

,text,label,label_enc
0,"['assisted living', 'interpersonal and communi...",skills,4
1,"['Python', 'Scikit Learn', 'Keras', 'Tensorflo...",skills,4
2,"['Data Analyst', 'Machine Learning', 'Data Min...",skills,4
3,"['Python', 'MySQL', 'Tensorflow', 'Keras', 'Ma...",skills,4
4,"['Java', 'Spring', 'Javascript', 'CSS', 'HTML'...",skills,4


In [62]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = balanced_df["text"].tolist()
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)
embeddings.shape

Python(42942) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10775.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 175/175 [00:16<00:00, 10.41it/s]


(11176, 384)

In [63]:
X_train, X_test, y_train, y_test = train_test_split(embeddings,
    balanced_df['label_enc'].values,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df['label_enc'].values)
print(f"Training set: {X_train.shape}. Testing set: {X_test.shape}")

Training set: (8940, 384). Testing set: (2236, 384)


In [65]:
log_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
log_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [67]:
y_pred = log_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

                 precision    recall  f1-score   support

 certifications       0.95      0.93      0.94       236
         degree       1.00      0.99      1.00       400
     experience       1.00      1.00      1.00       400
extracurricular       0.97      0.97      0.97       400
         skills       0.98      1.00      0.99       400
        summary       1.00      1.00      1.00       400

       accuracy                           0.99      2236
      macro avg       0.98      0.98      0.98      2236
   weighted avg       0.99      0.99      0.99      2236



In [69]:
import pickle

In [70]:
with open('resume_section_classifier.pkl', 'wb') as f:
    pickle.dump(log_model, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

In [74]:
import os
print(os.path.getsize('models/resume_section_classifier.pkl'))  # should be several MB
print(os.path.getsize('models/label_encoder.pkl'))

19215
319


In [72]:
with open('resume_section_classifier.pkl', 'wb') as f:
    pickle.dump(log_model, f)

In [73]:
print(os.path.getsize('resume_section_classifier.pkl'))

19215
